# Lab 6 — One U-Net, two targets

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kleinric/cv-labs/blob/main/lab-06.ipynb)

**COMS4036A / COMS7050A Computer Vision · Week 6**

**Due: Thursday 3 September 2026, 17:00.** Groups of up to three; one member submits the group's notebook (`.ipynb`) on Moodle. Every member must be able to explain every cell.

Companion reading is [Chapter 6 of the course book](https://courses.ms.wits.ac.za/~richard/cv/book/chapters/06-segmentation.html). Labs 4 and 5 asked where the objects were; this one asks which pixels they occupy. You build a U-Net from the chapter's description, train it to separate pet from background on the same Oxford-IIIT Pet photographs Lab 3 classified, and reach about 0.72 IoU in eight epochs on a network of under two million parameters. Then you point the same network at a different target and retrain it, changing four lines.

This week's rung of the skills ladder is **checkpointing and resume**: every epoch's model and optimiser state saved, logged to W&B as an artifact, and a run picked up again from that artifact after it is deliberately killed. Long runs die — the Colab session times out, the queue evicts you, someone trips over the cable — and a run that cannot be resumed is a run you have to start again.

The Friday session covers Sections 0–4; Sections 5–7 are the take-home half.

## 0. GPU, W&B, and the data

Open this notebook in [Colab](https://colab.research.google.com) — the badge above — and **File ▸ Save a copy in Drive**. Set **Runtime ▸ Change runtime type** to a GPU, and log in to W&B; this lab logs to a project called `cv-lab6`.

In [ ]:
# Group members — fill in before submitting.
MEMBERS = [
    # ("Student name", "Student number"),
]
for name, number in MEMBERS:
    print(f"{number}  {name}")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import wandb
wandb.login()

The data is Lab 3's download, both archives again — the images you have already classified, plus the `annotations/` directory you only read a text file from. This week the useful part of `annotations/` is `trimaps/`.

In [ ]:
!wget -q -nc https://thor.robots.ox.ac.uk/pets/images.tar.gz
!wget -q -nc https://thor.robots.ox.ac.uk/pets/annotations.tar.gz
!tar -xzf images.tar.gz && tar -xzf annotations.tar.gz
!ls annotations/trimaps | head -3

In [ ]:
import math
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms
from torchvision.transforms import functional as TF

DEV = "cuda" if torch.cuda.is_available() else "cpu"
SIZE = 128


def seed_everything(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(0)
print("device:", DEV)

## 1. The data, and the transform that has to agree with itself

`annotations/trimaps/Abyssinian_100.png` is a single-channel PNG the same size as `images/Abyssinian_100.jpg`, and every pixel of it holds one of three integers: **1** for the animal, **2** for the background, **3** for the boundary — a band a few pixels wide around the outline where the annotator would not commit. There are 7,390 of them, one per photograph in `images/`, and they were drawn by hand (Parkhi et al., 2012).

The target this lab predicts is binary: pet or not-pet. That leaves a decision about label 3, which you make once, write down, and keep.

The photograph and its mask go through one pipeline together, and it has to treat them consistently. Whatever geometry you apply to the photograph — a crop, a flip, a rotation — the mask gets the *same* geometry, chosen once per item and not sampled twice. Resampling is where the two part company: a photograph interpolates bilinearly, and a mask must not. An average of label 1 and label 3 is label 2, which is a lie about that pixel.

**Q1.1.** Write `PetSegDataset(split, size=128, train=False)` — Lab 3's `PetsDataset` with the trimap attached. `__getitem__(i)` opens photograph $i$ and its trimap and returns `(image, mask, trimap)`: the image as a float tensor in $[0,1]$ (`TF.to_tensor`; no normalisation this week), the binary pet mask as a float tensor of shape $(1, s, s)$, and the raw three-label trimap as a `long` tensor of shape $(s, s)$ for Q1.2 and Q1.4 to count.

- when `train=True`: resize both to $1.15s$, take one `transforms.RandomCrop.get_params` window and apply it to both with `TF.crop`, then flip both or neither;
- when `train=False`: resize both to $s \times s$ and stop, so evaluation is deterministic;
- images resize with `TF.InterpolationMode.BILINEAR`, trimaps with `NEAREST`.

Build `train_set` (from `trainval`, `train=True`), `test_set` (from `test`, `train=False`), a training loader with `batch_size=32, shuffle=True, num_workers=2, drop_last=True`, and a test loader with `batch_size=32`. Print the size of each split.

In [ ]:
class PetSegDataset(Dataset):
    # YOUR CODE HERE
    ...

**Q1.2.** Display eight training pairs as two rows — the photograph, and the photograph with its mask drawn over it (`plt.contour(mask, [0.5])` or a translucent overlay; anything you can read). Run the cell twice so you see two different crops of the same eight animals. Then report, over 200 training images, the fraction of pixels carrying each of the three trimap labels. Does every mask sit on its animal?

In [ ]:
# YOUR CODE HERE

*Answer:*

**Q1.3.** Break it on purpose. Copy your `__getitem__` into a throwaway function that draws the crop window for the image and the crop window for the mask from two *separate* calls to `get_params`, and display eight pairs from it. Then say what you see — and, more usefully, what a training run on this dataset would look like from the outside, given that nothing here raises an exception and the loss still falls.

In [ ]:
# YOUR CODE HERE

*Answer:*

**Q1.4.** Resample one trimap to $128 \times 128$ twice, once with `NEAREST` and once with `BILINEAR`, and report the percentage of pixels that end up with a different label. Separately, take the *binary* pet mask as a float tensor at full resolution, resize it bilinearly to $128 \times 128$, and print how many distinct values it holds. Then answer: what should a mask be resampled with, and what did you do with trimap label 3?

In [ ]:
# YOUR CODE HERE

*Answer:*

## 2. The metric, before the model

Two numbers describe a predicted binary mask against its target. **Pixel accuracy** is the fraction of pixels whose predicted label matches. **Intersection over union** is $|P \cap T| / |P \cup T|$ over the foreground: the pixels both call pet, divided by the pixels either calls pet.

Write them now, while there is nothing to be pleased about, and measure the trivial predictor with them before you measure your own.

**Q2.1.** Implement `pixel_accuracy(pred, target)` and `iou(pred, target)` for batches of shape $(N, 1, H, W)$ holding zeros and ones, each returning a tensor of $N$ per-image values. Check both on a case you compute first on paper: two $4 \times 4$ masks, one covering the top two rows and the other the middle two rows.

In [ ]:
def pixel_accuracy(pred, target):
    # YOUR CODE HERE
    ...


def iou(pred, target, eps=1e-7):
    # YOUR CODE HERE
    ...

**Q2.2.** Run the predictor that outputs background everywhere over the whole test split, and report its mean pixel accuracy and mean IoU. Report the mean fraction of pixels that are pet as well. What is pixel accuracy worth as the headline number for this task?

In [ ]:
# YOUR CODE HERE

*Answer:*

**Q2.3.** The union in the denominator can be zero. Count the test images whose mask holds no pet pixel at all at 128 px, display them next to their trimaps, and check whether their *full-resolution* trimaps have any pet pixels either — a mask can be empty because the animal is small and resampling lost it, or because nobody labelled it. Report what your `iou` returns for such an image; the `eps` above is a convention, and it is making a claim.

In [ ]:
# YOUR CODE HERE

*Answer:*

## 3. The U-Net

Chapter 6's description, at lab scale. An encoder of three blocks — 32, 64, 128 channels — each a `DoubleConv` (two $3 \times 3$ convolutions, each followed by batch norm and ReLU), with a $2 \times 2$ max-pool between blocks. A bottleneck `DoubleConv` at 256 channels. A decoder of three blocks that mirror the encoder: a $2 \times 2$ transposed convolution that doubles the resolution and halves the channels, then a **concatenation** of the encoder tensor at that resolution, then a `DoubleConv` back down to the encoder's channel count. Finally a $1 \times 1$ convolution to the output channels.

The concatenation is the whole point, and it is also the shape bug you will hit: the `DoubleConv` after `up3` takes 256 input channels, not 128, because 128 came up the decoder and 128 came across the skip.

**Q3.1.** Implement `DoubleConv` and `UNet(in_ch=3, out_ch=1, base=32)`. Give `forward` a `trace=False` argument that, when true, prints the shape of every encoder, bottleneck and decoder tensor — the cheapest debugger there is.

In [ ]:
class DoubleConv(nn.Module):
    # YOUR CODE HERE
    ...


class UNet(nn.Module):
    # YOUR CODE HERE
    ...

**Q3.2.** Before you run it: write down the shape you expect at each of the seven traced tensors for a $1 \times 3 \times 128 \times 128$ input, and compute the bottleneck block's parameter count by hand ($C_{\text{out}} \times C_{\text{in}} \times 9$ per convolution, no bias because batch norm follows, plus $2C$ per batch-norm layer). Then run the trace and `sum(p.numel() ...)` and check both. Report the total.

In [ ]:
# YOUR CODE HERE

*Answer:*

**Q3.3.** The ritual, from Lab 2 and expected before every real run since: take one batch of 16 training pairs and drive a fresh seeded `UNet` to memorise it — Adam at $10^{-3}$, 250 steps, `F.binary_cross_entropy_with_logits` against the mask. Log it to W&B under a name that makes clear it is the ritual. Print the loss and the batch IoU every 50 steps, and report the first step at which the batch IoU passes 0.99.

In [ ]:
# YOUR CODE HERE

## 4. Train it, kill it, resume it

A checkpoint is not the weights. Weights alone will restore what the network predicts and nothing about how it was moving: Adam carries a first and second moment for every parameter, and a run that resumes without them takes a few dozen steps to rebuild an estimate it already had. Save, every epoch: the model's `state_dict`, the optimiser's `state_dict`, the epoch number, and the random-number state. Then `wandb.log_artifact` it, so the file is attached to the run rather than to a virtual machine that is about to disappear.

Logging a checkpoint as a W&B **artifact** takes three lines:

```python
art = wandb.Artifact(f"unet-seg-{wandb.run.id}", type="model")
art.add_file("ckpt.pt")
wandb.log_artifact(art, aliases=[f"epoch-{epoch}"])
```

and getting it back in another run — another session, another machine — takes two:

```python
path = wandb.run.use_artifact(f"unet-seg-{run_id}:epoch-4", type="model").download()
ckpt = torch.load(os.path.join(path, "ckpt.pt"), weights_only=False)
```

**Q4.1.** Write `save_ckpt(path, model, opt, epoch)` and `load_ckpt(path, model, opt=None)`. `save_ckpt` stores the four things named above; `load_ckpt` restores the model, restores the optimiser when one is passed, restores the random-number state, and returns the epoch. Write `evaluate(model, loader)` returning mean pixel accuracy and mean IoU over a loader, and `run_epoch(model, opt, loader)` returning the mean training loss and the list of per-step losses.

In [ ]:
def save_ckpt(path, model, opt, epoch):
    # YOUR CODE HERE
    ...


def load_ckpt(path, model, opt=None):
    # YOUR CODE HERE
    ...


def evaluate(model, loader):
    # YOUR CODE HERE
    ...


def run_epoch(model, opt, loader):
    # YOUR CODE HERE
    ...

**Q4.2.** Seed, build a fresh `UNet`, and train for 8 epochs with Adam at $10^{-3}$ and batch size 32. Every epoch: evaluate, log `train/loss`, `val/acc` and `val/iou` to W&B, save a checkpoint, and log it as an artifact with an `epoch-N` alias. The config must carry the architecture's base width, the image size, the optimiser and learning rate, the batch size, the epoch count, the parameter count and the seed. Report the final pixel accuracy and IoU beside Q2.2's trivial predictor, and plot both against epoch.

In [ ]:
# YOUR CODE HERE

**Q4.3.** Now the rung. Pretend the run above died after epoch 4. Start a second W&B run, build a *fresh* `UNet` and a *fresh* Adam, pull the **epoch-4** artifact from Q4.2's run, load it, and train two more epochs — epochs 5 and 6, which Q4.2 also ran, so you have something to compare against. Do the whole thing twice: once passing the optimiser to `load_ckpt`, once not passing it. Three things to report.

1. The pixel accuracy and IoU straight after loading, before any training — compare against what Q4.2 logged at epoch 4.
2. The per-step training loss over the first 20 steps of each resume, plotted together. Both resumes see the same batches in the same order, because the random-number state came out of the checkpoint too, so the two curves are comparable step by step.
3. The mean loss over those 20 steps, and the mean loss over each of the two resumed epochs.

Then answer: what did the optimiser state buy, how long did the cost of dropping it last, and which of the three reported quantities would have caught the mistake if you had not been looking for it?

In [ ]:
# YOUR CODE HERE

*Answer:*

**Q4.4.** Nothing in your `UNet` names a resolution. Build test loaders at 64, 128 and 256 px — the same photographs and the same trimaps, resized differently — and evaluate the Q4.2 weights on all three, unchanged. Report accuracy and IoU at each. Then say what limits the network at 64 and what limits it at 256, given that the convolutions have not moved.

In [ ]:
# YOUR CODE HERE

*Answer:*

## 5. Someone else's segmenter

`maskrcnn_resnet50_fpn_v2` is torchvision's instance segmenter, trained on COCO: 118,000 images, 80 classes, of which two are `cat` (label 17) and `dog` (label 18). It has never seen this dataset. It returns, per image, a set of boxes, labels, scores and soft masks of shape $(N, 1, H, W)$.

In [ ]:
from torchvision.models.detection import (MaskRCNN_ResNet50_FPN_V2_Weights,
                                          maskrcnn_resnet50_fpn_v2)

mrcnn = maskrcnn_resnet50_fpn_v2(
    weights=MaskRCNN_ResNet50_FPN_V2_Weights.COCO_V1).to(DEV).eval()
print(f"{sum(p.numel() for p in mrcnn.parameters()):,} parameters")

**Q5.1.** Take a fixed seeded subset of 300 test images. Evaluate your Q4.2 network on it. Then run Mask R-CNN on the *same* 128 px tensors — it takes a list of images in $[0,1]$, and resizes internally — and turn its output into a binary pet mask: keep the detections labelled cat or dog with score above 0.5, threshold their soft masks at 0.5, and take the union. Report mean IoU for both, and the number of images in which Mask R-CNN found no cat and no dog at all.

In [ ]:
# YOUR CODE HERE

**Q5.2.** Find the six images where the two disagree most — rank by the absolute difference of their per-image IoU — and display each as a row: photograph, ground-truth mask, your mask, Mask R-CNN's mask. Describe the two failure modes you can see, one per model.

In [ ]:
# YOUR CODE HERE

*Answer:*

**Q5.3.** Mask R-CNN has 46 million parameters and was fitted on 118,000 annotated images of eighty everyday classes; yours has under two million and saw 3,680 photographs of cats and dogs for eight epochs. Compare the two IoU numbers you measured and say what they mean for the question "should I fine-tune a pretrained dense-prediction model or train one from scratch?" — and name the property of *this* dataset that makes your answer what it is.

*Answer:*

## 6. The same network, a different target

Nothing in the U-Net knows about pets. It is a function from an image to an image of the same size, and what it computes is set by the pair you train it on and the loss you score it with. Change the pair and the loss; leave the architecture alone.

The new pair is (noisy photograph, clean photograph). Add Gaussian noise of standard deviation $\sigma$ to an image in $[0,1]$, draw $\sigma$ per image from $\mathcal{U}(0.05, 0.4)$ so the network cannot specialise to one level, and clamp back to $[0,1]$. The target is the clean image. The loss is mean squared error. Quality is reported as **PSNR**, $10 \log_{10}(1/\text{MSE})$ for images in $[0,1]$ — decibels, higher is better, and 3 dB is a halving of the error.

**Q6.1.** Write `add_noise(x, sigma)` and display four training triples: clean, noisy, and the difference, at $\sigma$ = 0.05, 0.15, 0.25 and 0.4, with the input PSNR in each title. Look at 0.4 — that is what the network is being asked to undo.

In [ ]:
# YOUR CODE HERE

**Q6.2.** Build `UNet(in_ch=3, out_ch=3)`, seed, and train for 6 epochs with Adam at $10^{-3}$ on the same loader — noise drawn fresh per batch, MSE against the clean image. Log to W&B with a complete config, checkpoint each epoch, and log the final checkpoint as an artifact; a later lab uses these weights. Report the parameter count beside Section 4's network and the validation MSE at $\sigma = 0.2$ per epoch.

In [ ]:
# YOUR CODE HERE

**Q6.3.** For $\sigma \in \{0.1, 0.2, 0.4\}$ on the test split, report the PSNR of the noisy input and the PSNR of the network's output, and display four (noisy, output, clean) triples at $\sigma = 0.2$.

In [ ]:
# YOUR CODE HERE

**Q6.4.** Now $\sigma = 0.6$, which is outside the range it trained on. Report input and output PSNR, display four triples, and say what the output looks like — how it fails is more informative than that it fails.

In [ ]:
# YOUR CODE HERE

*Answer:*

**Q6.5.** List what changed between Section 4 and Section 6 and what did not — down to the line, and including the parameter count. A later chapter returns to this network.

*Answer:*

## 7. The record

Paste links to your W&B runs below. At minimum: the one-batch ritual, the eight-epoch segmentation run with its per-epoch checkpoint artifacts, both resumes, and the second-target run. Each needs a meaningful name and a complete config — base width, image size, optimiser, learning rate, batch size, epochs, parameter count, seed — and the artifact aliases must be visible on the segmentation run.

*W&B run links:*

## 8. Before you submit

- [ ] **Runtime ▸ Restart session and run all** on a GPU runtime, then read every output. The full re-run downloads 800 MB, trains three networks and resumes two; budget 45 minutes on a T4.
- [ ] Group members filled in; every member can explain every cell.
- [ ] Every *Answer:* cell answered; W&B links pasted in Section 7, with the checkpoint artifacts attached.
- [ ] **File ▸ Download ▸ Download .ipynb**, one member submits on Moodle before **Thursday 3 September, 17:00**.